### Complete Order (Delay)
For delay-based objectives, two separation-identical aircraft $i$ and $j$ induce a complete order when $r_i \leq r_j$ and $b_i \leq b_j$. For any partial sequence, placing $i$ before $j$ yields a schedule whose delay is no worse than the schedule obtained by swapping them, for every aircraft other than $i$ and $j$. The cell below verifies both the per-aircraft and total-delay variants of this dominance relation.


In [1]:
from core.context import *
from core.checks import *

# Construct the two sequences that differ only in the relative order of i and j.
S1, S2, ctx = get_sequences()

# State the premises: order the release and baseline times, and require i and j to be separation-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.b["i"] <= ctx.b["j"],   *ctx.separation_equivalence("i", "j"),
]



# Verify that every aircraft other than i and j incurs no larger delay in S1.
claim = And(*[
    S1.delay[x] <= S2.delay[x]
    for x in ctx.aircraft if x not in ("i", "j")
])

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Complete Order Delay (per-aircraft): {res}")



# Verify that total delay in S1 is no larger than in S2.
claim = Sum([S1.delay[x] for x in ctx.aircraft]) <= Sum([S2.delay[x] for x in ctx.aircraft])

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Complete Order Delay (total):        {res}")


Complete Order Delay (per-aircraft): Verified
Complete Order Delay (total):        Verified


### Complete Order (Makespan)
For makespan minimisation, two separation-identical aircraft $i$ and $j$ induce a complete order when $r_i \leq r_j$. In any partial sequence, placing $i$ before $j$ produces a makespan no worse than the one obtained by swapping their positions. The cell below verifies this dominance argument directly.

In [2]:
from core.context import *
from core.checks import *

# Construct the two sequences that differ only in the relative order of i and j.
S1, S2, ctx = get_sequences()

# State the premises: order the release times and require i and j to be separation-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],   *ctx.separation_equivalence("i", "j"),
]

# Verify that the makespan of S1 is no larger than that of S2.
claim = S1.makespan <= S2.makespan

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Complete Order Makespan: {res}")


Complete Order Makespan: Verified


### Complete Order (Time Window Feasibility)
For time-window feasibility, two separation-identical aircraft $i$ and $j$ induce a complete order when $r_i \leq r_j$ and $lt_i \leq lt_j$. If the sequence with $j$ before $i$ is feasible, then swapping them so that $i$ precedes $j$ does not create any new time-window violations. The cell below verifies this preservation result.


In [3]:
from core.context import *
from core.checks import *

# Construct the two sequences that differ only in the relative order of i and j.
S1, S2, ctx = get_sequences()

# State the premises: order the release and latest times, require separation equivalence, and assume S2 is feasible.
premises = [
    ctx.r["i"] <= ctx.r["j"],               ctx.lt["i"] <= ctx.lt["j"],
    *ctx.separation_equivalence("i", "j"),  *S2.time_window_feasible,
]

# Verify that S1 does not introduce any time-window violation absent from S2.
claim = And(*[
    Implies(S2.window_violation[x], S1.window_violation[x])
    for x in ctx.aircraft
])

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Complete Order Time Windows: {res}")


Complete Order Time Windows: Verified
